# 08 · Hybrid models (Step 5)

**Why a hybrid.** Phase 1 showed that a regularised LogReg on the 43 hand-built features is hard to beat. The brief's hypothesis is that manual summaries miss interactions between time, airport, carrier and recent delay dynamics. The hybrid tests exactly that: keep the ML model, and let a network learn only what it misses.

* **Wide & Deep (main).** logit = LogReg part (wide: the Phase 1 LogReg design matrix) + deep part (embeddings, numerics and, if it won Step 4, the GRU history encoder). The wide part is initialised from the LogReg fitted on the same fold's rows and trained jointly; the deep part's output layer starts at zero, so the untrained hybrid **is** the Phase 1 LogReg (tested in `tests/test_train.py`). Joint training avoids the stacking leak of a frozen in-sample offset.
* **Embeddings → XGBoost (secondary).** The MLP's carrier, origin and destination embeddings added as features to the Phase 1 XGBoost configuration. Only low-cardinality embeddings are transferred: a rare tail number's embedding would memorise its labels.
* **Search:** 12 configurations of learning rate, weight decay and the wide part's learning-rate scale; best re-scored with 5 seeds.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import aerocast_dl  # torch before XGBoost (OpenMP)
import numpy as np, pandas as pd
from IPython.display import Image, display
from aerocast_dl import config, pipeline
R, F = config.RESULTS_DIR, config.FIGURES_DIR
pd.set_option('display.precision', 4)


## 1. Search

In [ ]:
search = pd.read_csv(R / 'search_wide_deep.csv').sort_values('cv_pr_auc', ascending=False)
spec = pipeline.load_spec('wide_deep')
print('deep part:', 'MLP + GRU' if spec.seq_names else 'MLP', '| wide part: Phase 1 LogReg (C = 0.1)')
search

## 2. Does the hybrid add value over LogReg?

Paired by fold, against the model it starts from. A gain inside the seed SD, or in only two of four folds, is not evidence of value.

In [ ]:
cv = pd.concat([pipeline.phase1_cv(), pd.read_csv(R / 'cv_models.csv')], ignore_index=True)
cv = cv[~cv['model'].str.startswith('ablation:')].set_index('model')
lr = cv.loc['Phase 1 LogReg', pipeline.FOLD_COLS].astype(float)
rows = ['Phase 1 LogReg', 'Phase 1 XGBoost', 'mlp', 'gru', 'wide_deep', 'emb_xgb']
for m in ['wide_deep', 'emb_xgb']:
    d = cv.loc[m, pipeline.FOLD_COLS].astype(float) - lr
    print(f'{m} - LogReg per fold: {np.round(d.to_numpy(), 4)} | mean {d.mean():+.4f} | better in {(d > 0).sum()}/4 folds')
cv.loc[rows, ['cv_pr_auc_mean', 'cv_pr_auc_std', *pipeline.FOLD_COLS, 'seed_sd_mean_over_folds', 'n_params']]

In [ ]:
display(Image(F / 'cv_models.png', width=700))